# 코호트 분석
코호트 = 같은 시기에 첫 구매한 고객 그룹  

예시)  
2017-01 코호트: 2017년 1월에 처음 구매한 고객들  
→ 이 그룹이 2월에 몇 %가 재구매했나?  
→ 3월에는? 6개월 후에는?  

→ 시간에 따른 고객 유지율 변화를 추적

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.unicode_minus'] = False

conn = sqlite3.connect('../ecommerce.db')
os.makedirs('../output', exist_ok=True)

def query(sql):
    return pd.read_sql(sql, conn)

START_DATE    = '2017-01-01'
END_DATE      = '2018-08-31'
SNAPSHOT_DATE = '2018-09-01'

## 1. 데이터 준비

In [7]:
# 고객별 주문 월 추출
orders_cohort = query(f"""
    SELECT
        c.customer_unique_id,
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month
    FROM orders o
    JOIN customers c USING (customer_id)
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp BETWEEN '{START_DATE}' AND '{END_DATE}'
    GROUP BY c.customer_unique_id, order_month           --- 고객+월 기준 중복 제거
    ORDER BY c.customer_unique_id, order_month
""")

# 첫 구매 월 (코호트 기준)
first_order = (orders_cohort
               .groupby('customer_unique_id')['order_month']
               .min()
               .reset_index()
               .rename(columns={'order_month': 'cohort_month'}))

orders_cohort = orders_cohort.merge(first_order, on = 'customer_unique_id')

print(f"전체 고객 수: {orders_cohort['customer_unique_id'].nunique():,}명")
display(orders_cohort.head())

전체 고객 수: 93,104명


,customer_unique_id,order_month,cohort_month
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05,2018-05
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05,2018-05
2,0000f46a3911fa3c0805444483337064,2017-03,2017-03
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10,2017-10
4,0004aac84e0df4da2b147fca70cf8255,2017-11,2017-11


## 2. 코호트 인덱스 계산

In [9]:
orders_cohort['order_month'] = pd.to_datetime(orders_cohort['order_month'])
orders_cohort['cohort_month'] = pd.to_datetime(orders_cohort['cohort_month'])

# 코호트 인덱스 = 첫 구매로부터 몇 달 후인가
orders_cohort['cohort_index'] = (
    (orders_cohort['order_month'].dt.year - orders_cohort['cohort_month'].dt.year)
    + (orders_cohort['order_month'].dt.month - orders_cohort['cohort_month'].dt.month)
)

print("📊 코호트 인덱스 분포")
print(orders_cohort['cohort_index'].value_counts().sort_index())
display(orders_cohort.head(10))

📊 코호트 인덱스 분포
cohort_index
-10       11
-9        41
-8        34
-7        54
-6        63
-5        59
-4        62
-3        68
-2        55
-1        70
 0     93159
 1       445
 2       257
 3       180
 4       139
 5        85
 6        78
 7        40
 8        19
 9         5
 10        5
 11        1
Name: count, dtype: int64


,customer_unique_id,order_month,cohort_month,cohort_index
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-01,2018-05-01,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-01,2018-05-01,0
2,0000f46a3911fa3c0805444483337064,2017-03-01,2017-03-01,0
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-01,2017-10-01,0
4,0004aac84e0df4da2b147fca70cf8255,2017-11-01,2017-11-01,0
5,0004bd2a26a76fe21f786e4fbd80607f,2018-04-01,2018-04-01,0
6,00050ab1314c0e55a6ca13cf7181fecf,2018-04-01,2018-04-01,0
7,00053a61a98854899e70ed204dd4bafe,2018-02-01,2018-02-01,0
8,0005e1862207bf6ccc02e4228effd9a0,2017-03-01,2017-03-01,0
9,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-01,2018-03-01,0


전체 고객 수 < 코호트 인덱스 0 갯수 -> 이상함. 원인 확인해보기

In [12]:
# customer_id 하나에 customer_unique_id가 여러 개인 케이스 확인
dup_check = query("""
    SELECT
        customer_id,
        COUNT(DISTINCT customer_unique_id) AS unique_id_count
    FROM customers
    GROUP BY customer_id
    HAVING COUNT(DISTINCT customer_unique_id) > 1
""")
print(f"customer_id 하나에 unique_id 여러 개인 케이스: {len(dup_check)}건")
display(dup_check)

# customer_unique_id 하나에 customer_id가 여러 개인 케이스 확인
dup_check2 = query("""
    SELECT
        customer_unique_id,
        COUNT(DISTINCT customer_id) AS id_count
    FROM customers
    GROUP BY customer_unique_id
    HAVING COUNT(DISTINCT customer_id) > 1
""")
print(f"customer_unique_id 하나에 customer_id 여러 개인 케이스: {len(dup_check2)}건")
display(dup_check2)

customer_id 하나에 unique_id 여러 개인 케이스: 0건


,customer_id,unique_id_count


customer_unique_id 하나에 customer_id 여러 개인 케이스: 2997건


,customer_unique_id,id_count
0,00172711b30d52eea8b313a7f2cced02,2
1,004288347e5e88a27ded2bb23747066c,2
2,004b45ec5c64187465168251cd1c9c2f,2
3,0058f300f57d7b93c477a131a59b36c3,2
4,00a39521eb40f7012db50455bf083460,2
...,...,...
2992,ff36be26206fffe1eb37afd54c70e18b,3
2993,ff44401d0d8f5b9c54a47374eb48c1b8,2
2994,ff8892f7c26aa0446da53d01b18df463,2
2995,ff922bdd6bafcdf99cb90d7f39cea5b3,3
